# 019 — Afinação dos pesos do ensemble

Parte exatamente do ensemble 016: XGBoost 007 (`multi:softprob`, `hist`, 200 árvores, profundidade 5, taxa 0.1, semente 42) e MLP 015 (64→32, ReLU, Adam `lr=0.0001`, pesos inversos à frequência, 93 épocas). O controlo é 75/25. Compara 60/40, 70/30, 75/25, 80/20 e 85/15 pela Macro F1 interna. Depois reajusta ambos os modelos nos 80% completos e avalia a mistura escolhida nos 20% fixos.

## Protocolo comum

Divisão estratificada fixa 80/20 (`random_state=42`) e validação interna estratificada de 15% dos 80% de treino (`random_state=42`), como nas experiências anteriores. A preparação é ajustada só nas linhas de treino de cada fase e os pesos de classe são calculados só nos respetivos alvos de treino. Todas as escolhas usam Macro F1 interna. A validação fixa já foi consultada em várias experiências; serve apenas como comparação de desenvolvimento, não como avaliação final independente.

Para a submissão, reajustam-se a preparação e modelos novos nas 10.962 linhas rotuladas, recalculando pesos de classe e usando as épocas escolhidas internamente. O CSV copia `RowId` do exemplo oficial na ordem das 3.655 linhas de teste e valida cabeçalho, unicidade, classes inteiras 0–15 e `index=False`.

CSV: `submissions/submission_019_ensemble_pesos.csv`. A pontuação Kaggle fica em branco até ser submetido.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents]
                 if (p / "src" / "experiments_019_021.py").is_file())
sys.path.insert(0, str(repo_root))
from src.experiments_019_021 import run_019

result = run_019()
candidates = pd.DataFrame([
    {"XGBoost": row["xgboost_weight"], "MLP": row["mlp_weight"],
     "Macro F1 interna": row["internal_metrics"]["macro_f1"],
     "Accuracy interna": row["internal_metrics"]["accuracy"],
     "Controlo 75/25": (row["xgboost_weight"], row["mlp_weight"]) == (0.75, 0.25),
     "Selecionada": row is result["selected"]}
    for row in result["candidates"]
])
display(candidates.style.format({"XGBoost": "{:.0%}", "MLP": "{:.0%}",
                                 "Macro F1 interna": "{:.4f}", "Accuracy interna": "{:.4f}"}))
def show_metrics(metrics, title):
    print(title)
    print(f"Macro F1: {metrics['macro_f1']:.4f} | accuracy: {metrics['accuracy']:.4f}")
    per_band = pd.DataFrame(metrics["per_class"]).rename(
        columns={"classe": "Classe", "escalão": "Escalão", "F1": "F1", "recall": "Recall", "support": "Suporte"}
    )
    display(per_band.style.apply(
        lambda row: ["background-color: #fff2cc" if row["Classe"] in (1, 6, 13) else ""] * len(row),
        axis=1,
    ).format({"F1": "{:.4f}", "Recall": "{:.4f}"}))

show_metrics(result["fixed_metrics"], "Mistura selecionada — validação fixa")
print(f"Configuração selecionada: {result['configuration']}")
print(f"Diferença Macro F1 fixa vs. 016 (0.4580): {result['difference_from_016_fixed_macro_f1']:+.4f}")
print(f"Repete os pesos/configuração do 016: {result['repeats_016_configuration']}")
print(f"Previsões diferentes do CSV 016: {result['different_predictions_from_016']} / 3655; duplicado: {result['duplicate_016_predictions']}")
print(f"CSV: {result['submission']}")


,XGBoost,MLP,Macro F1 interna,Accuracy interna,Controlo 75/25,Selecionada
0,60%,40%,0.4986,0.5046,False,True
1,70%,30%,0.4854,0.4932,False,False
2,75%,25%,0.4859,0.4947,True,False
3,80%,20%,0.4871,0.4977,False,False
4,85%,15%,0.4802,0.4909,False,False


Mistura selecionada — validação fixa
Macro F1: 0.4557 | accuracy: 0.4528


,Classe,Escalão,F1,Recall,Suporte
0,0,15–17 anos,0.8235,0.8537,41
1,1,18–19 anos,0.4762,0.4412,34
2,2,20–24 anos,0.5926,0.6000,80
3,3,25–29 anos,0.3613,0.3733,75
4,4,30–34 anos,0.2410,0.2222,90
5,5,35–39 anos,0.3282,0.3333,129
6,6,40–44 anos,0.3771,0.3837,172
7,7,45–49 anos,0.3529,0.3295,173
8,8,50–54 anos,0.6094,0.6464,181
9,9,55–59 anos,0.6112,0.6158,203


Configuração selecionada: 60% XGBoost 007 + 40% MLP 015; MLP Adam lr=0.0001, 93 épocas
Diferença Macro F1 fixa vs. 016 (0.4580): -0.0023
Repete os pesos/configuração do 016: False
Previsões diferentes do CSV 016: 291 / 3655; duplicado: False
CSV: submissions/submission_019_ensemble_pesos.csv


## Conclusão e Decisão
- Data: 12/11
- Pontuação Kaggle: 0.48
- Conclusão: A mistura 60/40 igualou os 0,48 do 016, mas ficou abaixo na validação fixa (0,4557 vs. 0,4580). Alterar os pesos não trouxe uma melhoria clara.
- Melhorias: Manter os componentes e combinar várias sementes do MLP, fazendo a média das probabilidades antes da mistura com XGBoost. Pode reduzir a variabilidade.